# S13 – HuggingFace и BERT: токенизация, special tokens и входы модели

В этом ноутбуке разбираем **самую базовую и при этом критически важную** часть работы с трансформерами:  
как обычный текст превращается в набор входов, которые понимает BERT.

Цель ноутбука – снять эффект «чёрного ящика».  
После него должно быть понятно:

- зачем нужен `tokenizer`;
- чем отличаются `tokenize()`, `encode()` и полный вызов `tokenizer(...)`;
- откуда берутся `input_ids`, `attention_mask`, `token_type_ids`;
- какую роль играют `[CLS]`, `[SEP]`, `[PAD]`, `[UNK]`, `[MASK]`;
- что именно передаётся в `model(**inputs)`.

В качестве базовой модели используем **`bert-base-multilingual-cased`**: она достаточно стандартна, поддерживает русский текст и хорошо подходит для учебной демонстрации.


## 0. План

К концу ноутбука надо уметь:

1. Загружать `AutoTokenizer` для BERT-подобной модели.
2. Сравнивать исходный текст, токены и числовые идентификаторы токенов.
3. Понимать назначение `input_ids`, `attention_mask`, `token_type_ids`.
4. Объяснять роль special tokens: `[CLS]`, `[SEP]`, `[PAD]`, `[UNK]`, `[MASK]`.
5. Правильно использовать `padding`, `truncation` и `max_length`.
6. Разбирать различия между одним текстом и парой текстов на входе модели.
7. Собирать словарь входов, который можно передать в BERT.


## 1. Импорты и общие настройки

Ниже подключаем базовые библиотеки, фиксируем `seed`, выбираем устройство и загружаем токенизатор.

> Предполагается, что в окружении уже доступны `torch`, `transformers`, `pandas` и `numpy`.


In [ ]:

# Базовые библиотеки для воспроизводимости, анализа и удобного отображения результатов.
import random
from typing import Iterable

import numpy as np
import pandas as pd
import torch

from transformers import AutoModel, AutoTokenizer


In [ ]:

# Фиксируем seed и определяем устройство.
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Берём стандартную мультиязычную BERT-модель: она хорошо подходит для русскоязычных примеров.
MODEL_NAME = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded:", MODEL_NAME)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Model max length:", tokenizer.model_max_length)


Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: bert-base-multilingual-cased
Tokenizer class: BertTokenizer
Model max length: 512


## 2. Мини-набор текстов и постановка задачи

Нам нужны короткие учебные примеры, чтобы можно было руками разбирать токены и входы модели.  
Ниже задаём:

- несколько одиночных текстов;
- несколько пар текстов, чтобы потом увидеть, как появляются `token_type_ids`.


In [ ]:

# Небольшой набор одиночных текстов и пар текстов.
sample_texts = [
    "Сегодня хорошая погода, и модель работает стабильно.",
    "BERT умеет работать с токенами, а не с «сырыми» строками.",
    "Нейросети любят данные, но ошибки в подготовке данных любят ещё больше.",
    "E-mail support@example.com и ссылка https://example.org/test?q=1 тоже будут разбиваться на токены.",
    "ОченьОченьОченьДлинноеСловоБезПробелов иногда токенизируется на несколько частей.",
]

text_pairs = [
    ("Модель показала высокую точность на валидации.", "Но на реальных данных качество оказалось ниже."),
    ("Студент загрузил датасет.", "Затем он токенизировал тексты и передал их в BERT."),
    ("Это первый фрагмент.", "Это второй фрагмент."),
]

display(pd.DataFrame({"text": sample_texts}))
display(pd.DataFrame(text_pairs, columns=["text_a", "text_b"]))


,text
0,"Сегодня хорошая погода, и модель работает стаб..."
1,"BERT умеет работать с токенами, а не с «сырыми..."
2,"Нейросети любят данные, но ошибки в подготовке..."
3,E-mail support@example.com и ссылка https://ex...
4,ОченьОченьОченьДлинноеСловоБезПробелов иногда ...


,text_a,text_b
0,Модель показала высокую точность на валидации.,Но на реальных данных качество оказалось ниже.
1,Студент загрузил датасет.,Затем он токенизировал тексты и передал их в B...
2,Это первый фрагмент.,Это второй фрагмент.


## 3. Первое знакомство с `AutoTokenizer`

Начинаем с трёх базовых операций:

- `tokenize(text)` – вернуть список токенов;
- `encode(text)` – вернуть список id токенов;
- `decode(ids)` – восстановить строковое представление из id.

Важно: токены – это **не то же самое, что слова**.  
Один «человеческий» фрагмент текста может превратиться в несколько токенов.

BERT использует **WordPiece** – алгоритм разбиения на подслова.  
Если слово встречается в словаре целиком, оно остаётся одним токеном.  
Если нет – оно разбивается на части, причём все части **кроме первой** получают префикс `##`.

> **Пример:** слово `«токенами»` может выглядеть как `['током', '##ена', '##ми']`.  
> Префикс `##` означает: «этот кусок является продолжением предыдущего фрагмента».  
> В колонке `decoded_piece` таблицы ниже `##` убирается, и видна исходная подстрока.


In [ ]:

# Удобная вспомогательная функция для табличного просмотра токенизации.
def inspect_single_text(text: str, tokenizer) -> pd.DataFrame:
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    decoded_pieces = [tokenizer.decode([tid]) for tid in token_ids]
    return pd.DataFrame(
        {
            "position": list(range(len(tokens))),
            "token": tokens,
            "token_id": token_ids,
            "decoded_piece": decoded_pieces,
        }
    )

example_text = sample_texts[1]
print("Исходный текст:")
print(example_text)
print()

tokens = tokenizer.tokenize(example_text)
token_ids = tokenizer.encode(example_text, add_special_tokens=False)
decoded_text = tokenizer.decode(token_ids)

print("Токены:")
print(tokens)
print()

print("ID токенов:")
print(token_ids)
print()

print("decode(token_ids):")
print(decoded_text)

display(inspect_single_text(example_text, tokenizer))


Исходный текст:
BERT умеет работать с токенами, а не с «сырыми» строками.

Токены:
['BE', '##RT', 'ум', '##ее', '##т', 'работать', 'с', 'ток', '##ена', '##ми', ',', 'а', 'не', 'с', '«', 'с', '##ырым', '##и', '»', 'стр', '##ока', '##ми', '.']

ID токенов:
[46291, 46935, 39510, 38049, 10351, 45680, 558, 84964, 12868, 10508, 117, 541, 10375, 558, 208, 558, 91966, 10191, 220, 19240, 42075, 10508, 119]

decode(token_ids):
BERT умеет работать с токенами, а не с « сырыми » строками.


,position,token,token_id,decoded_piece
0,0,BE,46291,BE
1,1,##RT,46935,##RT
2,2,ум,39510,ум
3,3,##ее,38049,##ее
4,4,##т,10351,##т
5,5,работать,45680,работать
6,6,с,558,с
7,7,ток,84964,ток
8,8,##ена,12868,##ена
9,9,##ми,10508,##ми


## 4. Special tokens и служебная структура входа

BERT обычно работает не просто с последовательностью «обычных» токенов, а с последовательностью, куда автоматически добавляются специальные маркеры.

Самые важные:

- `[CLS]` – служебный токен в начале последовательности; часто используется для задач классификации;
- `[SEP]` – разделитель последовательностей;
- `[PAD]` – добивка до общей длины в батче;
- `[UNK]` – токен для неизвестных фрагментов;
- `[MASK]` – токен маскирования, важный для pretraining-задачи masked language modeling.

Ниже посмотрим и сами special tokens, и то, как они реально появляются в закодированном тексте.


In [ ]:

# Смотрим служебные токены токенизатора.
special_tokens_df = pd.DataFrame(
    {
        "name": [
            "cls_token",
            "sep_token",
            "pad_token",
            "unk_token",
            "mask_token",
        ],
        "token": [
            tokenizer.cls_token,
            tokenizer.sep_token,
            tokenizer.pad_token,
            tokenizer.unk_token,
            tokenizer.mask_token,
        ],
        "token_id": [
            tokenizer.cls_token_id,
            tokenizer.sep_token_id,
            tokenizer.pad_token_id,
            tokenizer.unk_token_id,
            tokenizer.mask_token_id,
        ],
    }
)

display(special_tokens_df)

# Смотрим полную последовательность с добавлением special tokens.
# Также запрашиваем token_type_ids, чтобы увидеть их для одиночного текста
# (они все равны 0 – контраст с парой текстов покажем в следующей секции).
encoded_with_specials = tokenizer(
    example_text,
    add_special_tokens=True,
    return_attention_mask=True,
    return_token_type_ids=True,
)

full_ids = encoded_with_specials["input_ids"]
full_tokens = tokenizer.convert_ids_to_tokens(full_ids)

display(
    pd.DataFrame(
        {
            "position": list(range(len(full_ids))),
            "token": full_tokens,
            "token_id": full_ids,
            "attention_mask": encoded_with_specials["attention_mask"],
            "token_type_id": encoded_with_specials["token_type_ids"],
        }
    )
)

print("Все token_type_ids для одного текста равны 0.")
print("Как они меняются для пары текстов – смотрим в секции 6.")


,name,token,token_id
0,cls_token,[CLS],101
1,sep_token,[SEP],102
2,pad_token,[PAD],0
3,unk_token,[UNK],100
4,mask_token,[MASK],103


,position,token,token_id,attention_mask,token_type_id
0,0,[CLS],101,1,0
1,1,BE,46291,1,0
2,2,##RT,46935,1,0
3,3,ум,39510,1,0
4,4,##ее,38049,1,0
5,5,##т,10351,1,0
6,6,работать,45680,1,0
7,7,с,558,1,0
8,8,ток,84964,1,0
9,9,##ена,12868,1,0


Все token_type_ids для одного текста равны 0.
Как они меняются для пары текстов — смотрим в секции 6.


## 5. `padding`, `truncation`, `max_length` и `attention_mask`

При работе с батчами последовательности обычно надо привести к совместимой длине.

Здесь важны три параметра:

- `padding` – добивает более короткие последовательности `[PAD]`-токенами;
- `truncation` – обрезает слишком длинные последовательности;
- `max_length` – задаёт верхнюю границу длины.

`attention_mask` показывает модели, какие позиции являются «настоящими» токенами, а какие – просто добивкой.


In [ ]:

# Подготовим короткий и длинный текст, чтобы наглядно увидеть padding и truncation.
short_text = "BERT понимает токены."
long_text = (
    "Трансформерная модель получает на вход последовательность токенов, "
    "дополненную служебными маркерами, а затем обрабатывает её механизмом внимания. "
    "Если текст слишком длинный, его часто приходится обрезать до max_length."
)

batch_no_trunc = tokenizer(
    [short_text, long_text],
    padding=True,
    return_tensors="pt",
)

batch_with_trunc = tokenizer(
    [short_text, long_text],
    padding=True,
    truncation=True,
    max_length=16,
    return_tensors="pt",
)

print("Без truncation:")
print("input_ids shape:", tuple(batch_no_trunc["input_ids"].shape))
print("attention_mask shape:", tuple(batch_no_trunc["attention_mask"].shape))
print()

print("С truncation и max_length=16:")
print("input_ids shape:", tuple(batch_with_trunc["input_ids"].shape))
print("attention_mask shape:", tuple(batch_with_trunc["attention_mask"].shape))
print()

for idx, text in enumerate([short_text, long_text]):
    print(f"--- Пример {idx} ---")
    print("Исходный текст:", text)
    print("Токены после truncation:")
    print(tokenizer.convert_ids_to_tokens(batch_with_trunc["input_ids"][idx].tolist()))
    print("attention_mask:")
    print(batch_with_trunc["attention_mask"][idx].tolist())
    print()

# Покажем padding в табличном виде на примере короткого текста из batch_no_trunc.
# Здесь хорошо видно: [PAD]-токены стоят после [SEP] и получают attention_mask=0.
pad_ids = batch_no_trunc["input_ids"][0].tolist()
pad_tokens = tokenizer.convert_ids_to_tokens(pad_ids)
pad_mask = batch_no_trunc["attention_mask"][0].tolist()

print("Короткий текст в батче с padding (batch_no_trunc, пример 0):")
display(
    pd.DataFrame(
        {
            "position": list(range(len(pad_ids))),
            "token": pad_tokens,
            "token_id": pad_ids,
            "attention_mask": pad_mask,
        }
    )
)
print("Позиции с [PAD] и attention_mask=0 – модель их игнорирует.")


Без truncation:
input_ids shape: (2, 66)
attention_mask shape: (2, 66)

С truncation и max_length=16:
input_ids shape: (2, 16)
attention_mask shape: (2, 16)

--- Пример 0 ---
Исходный текст: BERT понимает токены.
Токены после truncation:
['[CLS]', 'BE', '##RT', 'по', '##нима', '##ет', 'ток', '##ены', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
attention_mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]

--- Пример 1 ---
Исходный текст: Трансформерная модель получает на вход последовательность токенов, дополненную служебными маркерами, а затем обрабатывает её механизмом внимания. Если текст слишком длинный, его часто приходится обрезать до max_length.
Токены после truncation:
['[CLS]', 'Т', '##ран', '##с', '##фор', '##мер', '##ная', 'модель', 'получает', 'на', 'в', '##ход', 'после', '##дова', '##тельно', '[SEP]']
attention_mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Короткий текст в батче с padding (batch_no_trunc, пример 0):


,position,token,token_id,attention_mask
0,0,[CLS],101,1
1,1,BE,46291,1
2,2,##RT,46935,1
3,3,по,10297,1
4,4,##нима,97582,1
...,...,...,...,...
61,61,[PAD],0,0
62,62,[PAD],0,0
63,63,[PAD],0,0
64,64,[PAD],0,0


Позиции с [PAD] и attention_mask=0 — модель их игнорирует.


## 6. Один текст и пара текстов: `token_type_ids`

Для BERT важен сценарий, когда на вход подаются **две последовательности**.  
Например, в задачах entailment, similarity или answer selection.

В таких случаях токенизатор обычно возвращает ещё и `token_type_ids`:

- `0` – токены первой последовательности;
- `1` – токены второй последовательности.

Это один из способов сообщить модели, где заканчивается первый фрагмент и начинается второй.


In [ ]:

# Разбираем пару текстов и смотрим token_type_ids.
pair_a, pair_b = text_pairs[0]

pair_encoding = tokenizer(
    pair_a,
    pair_b,
    return_attention_mask=True,
    return_token_type_ids=True,
)

pair_ids = pair_encoding["input_ids"]
pair_tokens = tokenizer.convert_ids_to_tokens(pair_ids)
pair_mask = pair_encoding["attention_mask"]
pair_types = pair_encoding["token_type_ids"]

display(
    pd.DataFrame(
        {
            "position": list(range(len(pair_ids))),
            "token": pair_tokens,
            "token_id": pair_ids,
            "attention_mask": pair_mask,
            "token_type_id": pair_types,
        }
    )
)

print("Текст A:")
print(pair_a)
print()
print("Текст B:")
print(pair_b)


,position,token,token_id,attention_mask,token_type_id
0,0,[CLS],101,1,0
1,1,М,521,1,0
2,2,##оде,82562,1,0
3,3,##ль,12118,1,0
4,4,пока,45899,1,0
5,5,##зала,71742,1,0
6,6,вы,96195,1,0
7,7,##сок,45135,1,0
8,8,##ую,14611,1,0
9,9,точно,55819,1,0


Текст A:
Модель показала высокую точность на валидации.

Текст B:
Но на реальных данных качество оказалось ниже.


## 7. Батчевая токенизация

На практике мы почти всегда работаем не с одним примером, а с мини-батчем.  
Токенизатор умеет сразу принимать список строк и возвращать словарь тензоров согласованной формы.

Это уже очень близко к реальному сценарию перед подачей данных в модель.


In [ ]:

# Батчевая токенизация нескольких текстов.
batch = tokenizer(
    sample_texts,
    padding=True,
    truncation=True,
    max_length=24,
    return_tensors="pt",
)

print("Ключи batch:")
print(batch.keys())
print()

for key, value in batch.items():
    print(f"{key}: shape={tuple(value.shape)}")

# Для наглядности покажем длины последовательностей до и после padding/truncation.
raw_lengths = [len(tokenizer.encode(text, add_special_tokens=True)) for text in sample_texts]
real_token_count = batch["attention_mask"].sum(dim=1).tolist()

display(
    pd.DataFrame(
        {
            "text": sample_texts,
            "raw_length_with_specials": raw_lengths,
            "real_token_count": real_token_count,
        }
    )
)


Ключи batch:
KeysView({'input_ids': tensor([[   101,  52203,  10990,  50421,  91690,  43155,  14655,  10297,  77211,
            117,    549,  41344,  45544,  15888,  86840,  19619,    119,    102,
              0,      0,      0,      0,      0,      0],
        [   101,  46291,  46935,  39510,  38049,  10351,  45680,    558,  84964,
          12868,  10508,    117,    541,  10375,    558,    208,    558,  91966,
          10191,    220,  19240,  42075,  10508,    102],
        [   101,  21124,  10384,  44181,  37616,    552,  10593,  12528,  13081,
          41065,    117,  11279,    555,  13523,  67725,    543,  11429, 109120,
          34103,    552,  10593,  12528,  13081,    102],
        [   101,    142,    118,  30049,  13145,    137,  14351,    119,  10212,
            549,  90795,  14120,    131,    120,    120,  14351,    119,  10733,
            120,  15839,    136,    185,    134,    102],
        [   101,    523,  63549,  18002,  63549,  18002,  63549,  22681,  20505,
   

,text,raw_length_with_specials,real_token_count
0,"Сегодня хорошая погода, и модель работает стаб...",18,18
1,"BERT умеет работать с токенами, а не с «сырыми...",25,24
2,"Нейросети любят данные, но ошибки в подготовке...",27,24
3,E-mail support@example.com и ссылка https://ex...,34,24
4,ОченьОченьОченьДлинноеСловоБезПробелов иногда ...,29,24


## 8. Что реально получает BERT на вход

Теперь сделаем последний шаг: загрузим саму BERT-модель без classification head и передадим ей батч.  
Нас сейчас интересуют не предсказания классов, а **форма входов и форма выходов**.

Для базовой модели наиболее важны:

- `last_hidden_state` – представления всех токенов после последнего слоя;  
  форма: `[batch_size, seq_len, hidden_size]`, где для `bert-base-*` значение `hidden_size = 768`.
- `pooler_output` – агрегированное представление последовательности на основе `[CLS]`-токена  
  (если модель его возвращает); форма: `[batch_size, hidden_size]`.

> Размерность `768` – это архитектурный параметр `bert-base-*` (12 слоёв, 768 скрытых нейронов, 12 голов внимания).  
> У `bert-large-*` она равна `1024`. Её часто называют «embedding dimension» или «hidden dim».

Так становится ясно, что токенизатор и модель – это две разные, но тесно связанные части пайплайна.


In [ ]:

# Загружаем базовую BERT-модель и делаем один прямой проход без обучения.
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

model_inputs = {
    "input_ids": batch["input_ids"][:2].to(device),
    "attention_mask": batch["attention_mask"][:2].to(device),
}

# token_type_ids есть не у всех моделей, но для BERT они поддерживаются.
if "token_type_ids" in batch:
    model_inputs["token_type_ids"] = batch["token_type_ids"][:2].to(device)

with torch.no_grad():
    outputs = model(**model_inputs)

print("Ключи выхода модели:")
print(outputs.keys())
print()

print("Форма input_ids:", tuple(model_inputs["input_ids"].shape))
print("Форма attention_mask:", tuple(model_inputs["attention_mask"].shape))
if "token_type_ids" in model_inputs:
    print("Форма token_type_ids:", tuple(model_inputs["token_type_ids"].shape))
print()

print("Форма last_hidden_state:", tuple(outputs.last_hidden_state.shape))

if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
    print("Форма pooler_output:", tuple(outputs.pooler_output.shape))


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ключи выхода модели:
odict_keys(['last_hidden_state', 'pooler_output'])

Форма input_ids: (2, 24)
Форма attention_mask: (2, 24)
Форма token_type_ids: (2, 24)

Форма last_hidden_state: (2, 24, 768)
Форма pooler_output: (2, 768)


## 9. Типичные ошибки при токенизации и подготовке входов

Ниже – несколько антипримеров, которые полезно увидеть заранее.

1. **Ожидать, что число токенов равно числу слов.** Это неверно: токенизация работает по правилам конкретной модели.
2. **Ставить слишком маленький `max_length`.** Тогда модель будет терять значимую часть текста.
3. **Игнорировать special tokens.** Для BERT они важны и входят в нормальный формат входа.
4. **Думать, что `attention_mask` – необязательная мелочь.** На батчах с padding она принципиальна.


In [ ]:

# Антипример 1: число слов и число токенов почти никогда не совпадают.
tricky_text = "support@example.com обработан моделью за 0.23 секунды"

word_count = len(tricky_text.split())
token_count = len(tokenizer.tokenize(tricky_text))

print("Текст:", tricky_text)
print("Число слов по split():", word_count)
print("Число токенов tokenizer.tokenize():", token_count)
print("Токены:", tokenizer.tokenize(tricky_text))
print()

# Антипример 2: слишком агрессивное ограничение длины.
too_short = tokenizer(
    long_text,
    truncation=True,
    max_length=8,
    return_attention_mask=True,
)

print("Слишком короткий max_length=8:")
print(tokenizer.convert_ids_to_tokens(too_short["input_ids"]))
print()

# Антипример 3: отключение special tokens меняет ожидаемую структуру входа.
without_specials = tokenizer(example_text, add_special_tokens=False)
with_specials = tokenizer(example_text, add_special_tokens=True)

print("Длина без special tokens:", len(without_specials["input_ids"]))
print("Длина с special tokens:", len(with_specials["input_ids"]))
print("Начало последовательности с special tokens:")
print(tokenizer.convert_ids_to_tokens(with_specials["input_ids"][:8]))


Текст: support@example.com обработан моделью за 0.23 секунды
Число слов по split(): 6
Число токенов tokenizer.tokenize(): 17
Токены: ['support', '@', 'example', '.', 'com', 'об', '##ра', '##бо', '##тан', 'модель', '##ю', 'за', '0', '.', '23', 'секунд', '##ы']

Слишком короткий max_length=8:
['[CLS]', 'Т', '##ран', '##с', '##фор', '##мер', '##ная', '[SEP]']

Длина без special tokens: 23
Длина с special tokens: 25
Начало последовательности с special tokens:
['[CLS]', 'BE', '##RT', 'ум', '##ее', '##т', 'работать', 'с']


## 10. Итоги

1. **Токенизатор – это обязательный этап перед моделью.** BERT не работает с «сырыми» строками напрямую.
2. **Токены не совпадают со словами.** Один текст может разбиваться на части неожиданным образом.
3. **`input_ids`, `attention_mask`, `token_type_ids` – это нормальный формат входа для BERT-подобных моделей.**
4. **Special tokens входят в стандартную структуру последовательности.** Особенно важны `[CLS]` и `[SEP]`.
5. **`padding` и `truncation` нужны для практической работы с батчами.**
6. **После токенизации данные уже готовы к передаче в `model(**inputs)`.**


## Задания для самостоятельной работы

1. Возьмите 5 собственных русскоязычных текстов и сравните их токенизацию для `bert-base-multilingual-cased` и другой BERT-подобной модели.
2. Проверьте, как меняются `input_ids` и `attention_mask`, если поставить `max_length=12`, `24`, `48`.
3. Найдите примеры слов, URL, чисел или сокращений, которые разбиваются на неожиданное число токенов.
4. Токенизируйте 3 пары текстов и вручную разберите, где в `token_type_ids` проходит граница между первой и второй последовательностью.
5. Попробуйте передать в модель батч из двух текстов разной длины и проверьте формы входов и выходов.
